# Download CNN/DailyMail Stage 2 Part 2

In [ ]:
import json
import os
import random
import re
import time
import unicodedata
from pathlib import Path
from urllib.parse import urlencode
from urllib.request import urlopen

DATASET_NAME = "abisee/cnn_dailymail"
DATASET_CONFIG = "3.0.0"
DATASET_SPLIT = "train"
DATASET_ROWS_API = "https://datasets-server.huggingface.co/rows"

STAGE1_RAW_PATH = Path("data/raw/raw_news_stage1.jsonl")
STAGE2_PART1_RAW_PATH = Path("data/raw/raw_news_stage2_01.jsonl")
RAW_OUTPUT_PATH = Path("data/raw/raw_news_stage2_02.jsonl")
PROCESSED_OUTPUT_PATH = Path("data/processed/processed_news_02.jsonl")
SAMPLE_SIZE = 1000
MAX_ARTICLE_WORDS_EXCLUSIVE = 500
SEED = 43
SHUFFLE_BUFFER_SIZE = 10_000
OVERWRITE = False

WORD_RE = re.compile(r"\b\w+(?:[’'-]\w+)*\b", flags=re.UNICODE)
OUTPUT_COLUMNS = ["article_id", "article", "summary", "article_word_count"]
RAW_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
PROCESSED_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
random.seed(SEED)

print(f"Raw output:       {RAW_OUTPUT_PATH}")
print(f"Processed output: {PROCESSED_OUTPUT_PATH}")
print(f"Target: {SAMPLE_SIZE} new articles with < {MAX_ARTICLE_WORDS_EXCLUSIVE} words")

Raw output:       data\raw\raw_news_stage2_02.jsonl
Processed output: data\processed\processed_news_02.jsonl
Target: 500 new articles with < 500 words


## Load exclusions and define preprocessing

In [2]:
def clean_text(text: str, preserve_newlines: bool = False) -> str:
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text)
    if preserve_newlines:
        lines = [re.sub(r"[ \t]+", " ", line).strip() for line in text.split("\n")]
        return "\n".join(line for line in lines if line)
    return re.sub(r"\s+", " ", text).strip()

def count_words(text: str) -> int:
    return len(WORD_RE.findall(text))

def canonical_article(text: str) -> str:
    return clean_text(text).casefold()

def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        raise FileNotFoundError(f"Required dataset not found: '{path}'")
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON on line {line_number} of '{path}'.") from error
            if not record.get("article_id") or not record.get("article"):
                raise ValueError(f"Line {line_number} of '{path}' is missing article_id or article.")
            records.append(record)
    if not records:
        raise ValueError(f"'{path}' contains no records.")
    return records

stage1_records = load_jsonl(STAGE1_RAW_PATH)
stage2_part1_records = load_jsonl(STAGE2_PART1_RAW_PATH)
excluded_records = stage1_records + stage2_part1_records
excluded_ids = {str(record["article_id"]).strip() for record in excluded_records}
excluded_articles = {canonical_article(record["article"]) for record in excluded_records}

print(f"Loaded {len(stage1_records):,} Stage 1 records")
print(f"Loaded {len(stage2_part1_records):,} Stage 2 part 1 records")
print(f"Excluded unique IDs: {len(excluded_ids):,}")
print(f"Excluded unique normalized articles: {len(excluded_articles):,}")

Loaded 5,500 Stage 1 records
Loaded 500 Stage 2 part 1 records
Excluded unique IDs: 6,000
Excluded unique normalized articles: 5,999


## Download and select 500 non-overlapping records

In [ ]:
def fetch_dataset_page(offset: int, length: int = 100, retries: int = 4) -> dict:
    query = urlencode({
        "dataset": DATASET_NAME, "config": DATASET_CONFIG,
        "split": DATASET_SPLIT, "offset": offset, "length": length,
    })
    request_url = f"{DATASET_ROWS_API}?{query}"
    for attempt in range(1, retries + 1):
        try:
            with urlopen(request_url, timeout=120) as response:
                return json.loads(response.read().decode("utf-8"))
        except Exception:
            if attempt == retries:
                raise
            time.sleep(2 ** (attempt - 1))

def iter_dataset_rows():
    try:
        from datasets import load_dataset
        dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=DATASET_SPLIT, streaming=True).shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER_SIZE)
        print("Download method: datasets.load_dataset (streaming)")
        yield from dataset
        return
    except Exception as error:
        print(f"Streaming unavailable ({type(error).__name__}); using the rows API.")

    metadata = fetch_dataset_page(offset=0, length=1)
    page_size = min(metadata["num_rows_per_page"], 100)
    offsets = list(range(0, metadata["num_rows_total"], page_size))
    random.Random(SEED).shuffle(offsets)
    for offset in offsets:
        page = fetch_dataset_page(offset=offset, length=page_size)
        for wrapped_row in page["rows"]:
            if not wrapped_row.get("truncated_cells"):
                yield wrapped_row["row"]

raw_records, processed_records = [], []
selected_ids, selected_articles = set(), set()
scanned = rejected_overlap = rejected_length = rejected_missing = 0

for row in iter_dataset_rows():
    scanned += 1
    article_id = str(row.get("id") or "").strip()
    article = str(row.get("article") or "").strip()
    summary = str(row.get("highlights") or "").strip()
    processed_article = clean_text(article)
    processed_summary = clean_text(summary, preserve_newlines=True)
    article_key = processed_article.casefold()

    if not article_id or not processed_article or not processed_summary:
        rejected_missing += 1
        continue
    if (article_id in excluded_ids or article_key in excluded_articles or article_id in selected_ids or article_key in selected_articles):
        rejected_overlap += 1
        continue

    raw_word_count = count_words(article)
    processed_word_count = count_words(processed_article)
    if (raw_word_count >= MAX_ARTICLE_WORDS_EXCLUSIVE
            or processed_word_count >= MAX_ARTICLE_WORDS_EXCLUSIVE):
        rejected_length += 1
        continue

    raw_records.append({
        "article_id": article_id, "article": article, "summary": summary,
        "article_word_count": raw_word_count,
    })
    processed_records.append({
        "article_id": clean_text(article_id), "article": processed_article,
        "summary": processed_summary, "article_word_count": processed_word_count,
    })
    selected_ids.add(article_id)
    selected_articles.add(article_key)
    if len(raw_records) == SAMPLE_SIZE:
        break

if len(raw_records) != SAMPLE_SIZE:
    raise RuntimeError(f"Found only {len(raw_records)} qualifying articles after scanning {scanned:,} rows.")

print(f"Rows scanned: {scanned:,}")
print(f"Rejected overlaps: {rejected_overlap:,}")
print(f"Rejected at or above 500 words: {rejected_length:,}")
print(f"Rejected missing text: {rejected_missing:,}")
print(f"Collected: {len(raw_records):,}")

Streaming unavailable (ModuleNotFoundError); using the rows API.


Rows scanned: 1,446
Rejected overlaps: 31
Rejected at or above 500 words: 915
Rejected missing text: 0
Collected: 500


## Save both outputs

In [ ]:
if not OVERWRITE:
    existing = [path for path in (RAW_OUTPUT_PATH, PROCESSED_OUTPUT_PATH) if path.exists()]
    if existing:
        raise FileExistsError("Output already exists: " + ", ".join(map(str, existing)))

def write_temporary_jsonl(path: Path, records: list[dict]) -> Path:
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    with temporary_path.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")
    return temporary_path

raw_temp = RAW_OUTPUT_PATH.with_suffix(RAW_OUTPUT_PATH.suffix + ".tmp")
processed_temp = PROCESSED_OUTPUT_PATH.with_suffix(PROCESSED_OUTPUT_PATH.suffix + ".tmp")
try:
    raw_temp = write_temporary_jsonl(RAW_OUTPUT_PATH, raw_records)
    processed_temp = write_temporary_jsonl(PROCESSED_OUTPUT_PATH, processed_records)
    os.replace(raw_temp, RAW_OUTPUT_PATH)
    os.replace(processed_temp, PROCESSED_OUTPUT_PATH)
finally:
    for temporary_path in (raw_temp, processed_temp):
        if temporary_path.exists():
            temporary_path.unlink()

counts = [record["article_word_count"] for record in processed_records]
print(f"Saved {len(raw_records)} raw records to '{RAW_OUTPUT_PATH}'")
print(f"Saved {len(processed_records)} processed records to '{PROCESSED_OUTPUT_PATH}'")
print(f"Processed word-count range: {min(counts)} to {max(counts)}")

Saved 500 raw records to 'data\raw\raw_news_stage2_02.jsonl'
Saved 500 processed records to 'data\processed\processed_news_02.jsonl'
Processed word-count range: 68 to 499
